In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
# connect to your mart
engine = create_engine(f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}")

In [ ]:
# Analysis 1 — admission rate by chief complaint category
query = """
SELECT
    cc.complaint_category,
    COUNT(*) AS total_visits,
    SUM(CASE WHEN e.was_admitted THEN 1 ELSE 0 END) AS admissions,
    ROUND(100.0 * SUM(CASE WHEN e.was_admitted THEN 1 ELSE 0 END) / COUNT(*), 1) AS admission_rate_pct
FROM mrt.fct_ed_visits e
JOIN mrt.bridge_triage_complaints b ON e.stay_id = b.stay_id
JOIN mrt.dim_chiefcomplaint cc ON b.complaint_id = cc.complaint_id
GROUP BY cc.complaint_category
ORDER BY admission_rate_pct DESC
"""

In [ ]:
# Analysis 2 — average length of stay by acuity level
query2 = """
SELECT
    acuity,
    COUNT(*) AS visits,
    ROUND(AVG(length_of_stay_hours), 2) AS avg_los_hours
FROM mrt.fct_ed_visits
WHERE acuity IS NOT NULL
GROUP BY acuity
ORDER BY acuity
"""

In [ ]:
# Analysis 3 — ED visit volume by time of day
query3 = """
SELECT
    h.time_of_day,
    h.shift_type,
    COUNT(*) AS visit_count
FROM mrt.fct_ed_visits e
JOIN mrt.dim_hour h ON e.arrival_hour = h.time_hhmm
GROUP BY h.time_of_day, h.shift_type
ORDER BY visit_count DESC
"""